# Week 10: Matrix Factorization Recommender (SVD)

This notebook builds the advanced recommendation system for Week 10 using Singular Value Decomposition (SVD)
on the user-item rating matrix from MovieLens 25M.

Goal for this step:
- build a user-item rating matrix from the processed ratings
- apply truncated SVD (collaborative filtering via matrix factorization)
- generate top-N recommendations for each query movie using item-factor similarity
- save all outputs and model metadata for comparison in the evaluation notebook

This is a model-based collaborative filtering approach. Unlike content-based methods, it discovers
latent user preference patterns from rating behavior rather than from item features.

## Model design

We factorize the centered user-item rating matrix using truncated SVD:

```
R_centered ≈ U · Σ · Vᵀ
```

where V contains item latent factors. We use item-factor row vectors (Vᵀ rows = item vectors)
for item-to-item recommendation: a movie's row in V is its learned collaborative embedding.
Cosine similarity between these vectors gives us the collaborative filtering score.

Why SVD over explicit Surprise/ALS:
- reproducible without external matrix solvers
- `scipy.sparse` + `sklearn.decomposition.TruncatedSVD` handles the 25M-rating matrix efficiently
- the resulting item factors are well-understood and directly interpretable

In [1]:
from pathlib import Path

import json
import time

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
if not (project_root / 'data').exists():
    project_root = project_root.parent

ARTIFACTS_DIR = project_root / 'artifacts' / 'week10'
DATA_DIR = project_root / 'data' / 'processed' / 'week03_v1'
RANDOM_STATE = 42

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

required = [
    DATA_DIR / 'ratings_clean.parquet',
    DATA_DIR / 'movies_catalog.parquet',
    ARTIFACTS_DIR / 'week10_baseline_meta.json',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required inputs: {missing}')

ARTIFACTS_DIR

PosixPath('/Users/jay17/Documents/Proyects/big-data-tf/artifacts/week10')

## 1) Load ratings and build the user-item matrix

We load the full 25M ratings and build a sparse user-item matrix.
Ratings are mean-centered per user (subtract each user's mean) to remove individual rating biases.
Movies with fewer than 50 ratings and users with fewer than 20 ratings are excluded to reduce noise.

In [2]:
ratings_pl = pl.read_parquet(DATA_DIR / 'ratings_clean.parquet')
catalog_pl = pl.read_parquet(DATA_DIR / 'movies_catalog.parquet')
title_map = dict(zip(catalog_pl['movieId'].to_list(), catalog_pl['title'].to_list()))

MIN_MOVIE_RATINGS = 50
MIN_USER_RATINGS = 20

# Filter active movies
movie_counts = (
    ratings_pl
    .group_by('movieId')
    .agg(pl.len().alias('n'))
    .filter(pl.col('n') >= MIN_MOVIE_RATINGS)
)
active_movies = set(movie_counts['movieId'].to_list())

# Filter active users
user_counts = (
    ratings_pl
    .group_by('userId')
    .agg(pl.len().alias('n'))
    .filter(pl.col('n') >= MIN_USER_RATINGS)
)
active_users = set(user_counts['userId'].to_list())

# Filter ratings to active users and movies
ratings_filtered = (
    ratings_pl
    .filter(
        pl.col('movieId').is_in(list(active_movies))
        & pl.col('userId').is_in(list(active_users))
    )
)

print(f'Original ratings: {ratings_pl.height:,}')
print(f'After filtering (>={MIN_MOVIE_RATINGS} ratings/movie, >={MIN_USER_RATINGS} ratings/user): {ratings_filtered.height:,}')
print(f'Active movies: {ratings_filtered["movieId"].n_unique():,}')
print(f'Active users: {ratings_filtered["userId"].n_unique():,}')

Original ratings: 25,000,095
After filtering (>=50 ratings/movie, >=20 ratings/user): 24,644,928
Active movies: 13,176
Active users: 162,540


In [3]:
# Mean-center per user to remove user rating bias
user_means = (
    ratings_filtered
    .group_by('userId')
    .agg(pl.col('rating').mean().alias('user_mean_rating'))
)

ratings_centered = (
    ratings_filtered
    .join(user_means, on='userId', how='left')
    .with_columns([
        (pl.col('rating') - pl.col('user_mean_rating')).alias('rating_centered'),
    ])
)

# Build integer indices for sparse matrix
unique_users = sorted(ratings_centered['userId'].unique().to_list())
unique_movies = sorted(ratings_centered['movieId'].unique().to_list())

user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
movie_to_idx = {mid: i for i, mid in enumerate(unique_movies)}
idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}

n_users = len(unique_users)
n_movies = len(unique_movies)

print(f'Matrix dimensions: {n_users:,} users × {n_movies:,} movies')

# Build sparse CSR matrix
row_idxs = [user_to_idx[u] for u in ratings_centered['userId'].to_list()]
col_idxs = [movie_to_idx[m] for m in ratings_centered['movieId'].to_list()]
vals = ratings_centered['rating_centered'].to_list()

R_sparse = csr_matrix(
    (np.array(vals, dtype=np.float32), (row_idxs, col_idxs)),
    shape=(n_users, n_movies),
)

density = R_sparse.nnz / (n_users * n_movies)
print(f'Sparse matrix density: {density:.6f} ({R_sparse.nnz:,} non-zero entries)')

Matrix dimensions: 162,540 users × 13,176 movies
Sparse matrix density: 0.011508 (24,644,928 non-zero entries)


## 2) Truncated SVD — latent factor sweep

We sweep the number of latent factors from 10 to 150 and record the explained variance ratio.
This informs the choice of latent dimension for the final model.
The chosen dimension balances reconstruction quality with computational efficiency.

In [4]:
sweep_dims = [10, 20, 30, 50, 75, 100, 150]
sweep_results = []

for n_components in sweep_dims:
    t0 = time.time()
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
    svd.fit(R_sparse)
    elapsed = time.time() - t0
    explained = float(svd.explained_variance_ratio_.sum())
    print(f'  n_components={n_components:>3d}: explained_variance={explained:.4f}, time={elapsed:.1f}s')
    sweep_results.append({
        'n_components': n_components,
        'explained_variance': explained,
        'elapsed_seconds': round(elapsed, 2),
    })

sweep_df = pl.DataFrame(sweep_results)
sweep_df.write_csv(ARTIFACTS_DIR / 'week10_svd_sweep.csv')
display(sweep_df.to_pandas())

  n_components= 10: explained_variance=0.0933, time=1.4s
  n_components= 20: explained_variance=0.1227, time=2.1s
  n_components= 30: explained_variance=0.1442, time=2.1s
  n_components= 50: explained_variance=0.1779, time=2.8s
  n_components= 75: explained_variance=0.2111, time=3.6s
  n_components=100: explained_variance=0.2389, time=4.6s
  n_components=150: explained_variance=0.2858, time=5.4s


,n_components,explained_variance,elapsed_seconds
0,10,0.093265,1.35
1,20,0.122719,2.11
2,30,0.144218,2.07
3,50,0.177907,2.83
4,75,0.211054,3.57
5,100,0.238939,4.60
6,150,0.285809,5.37


In [5]:
fig_svd = go.Figure()
fig_svd.add_trace(go.Scatter(
    x=sweep_df['n_components'].to_list(),
    y=sweep_df['explained_variance'].to_list(),
    mode='lines+markers',
    marker=dict(size=8, color='#f59e0b'),
    line=dict(width=2.5, color='#f59e0b'),
    name='Explained variance',
))
fig_svd.update_layout(
    title='SVD latent factor sweep — cumulative explained variance',
    xaxis_title='Number of latent factors',
    yaxis_title='Explained variance ratio',
    height=450,
    template='plotly_white',
)
fig_svd.write_html(ARTIFACTS_DIR / 'week10_svd_sweep.html')
fig_svd.write_image(ARTIFACTS_DIR / 'week10_svd_sweep.png', scale=2)
fig_svd.show()

## 3) Train the final SVD model

We select 50 latent factors as the operating point.
This gives a good balance between explained variance and recommendation quality
without over-fitting to noise in the sparse matrix.

The item latent factor matrix (Vᵀ rows) is extracted and L2-normalized for cosine-similarity retrieval.

In [6]:
CHOSEN_COMPONENTS = 50

t0 = time.time()
svd_final = TruncatedSVD(n_components=CHOSEN_COMPONENTS, random_state=RANDOM_STATE)
U_factors = svd_final.fit_transform(R_sparse)    # shape: (n_users, n_components)
V_factors = svd_final.components_.T              # shape: (n_movies, n_components)  = item latent factors
elapsed = time.time() - t0

explained_final = float(svd_final.explained_variance_ratio_.sum())
print(f'Final SVD: n_components={CHOSEN_COMPONENTS}, explained_variance={explained_final:.4f}, time={elapsed:.1f}s')
print(f'U_factors shape (users): {U_factors.shape}')
print(f'V_factors shape (items): {V_factors.shape}')

# L2-normalize item factor rows for cosine similarity
V_normed = normalize(V_factors, norm='l2')

# Save item factors as parquet
factor_cols = [f'svd_{i+1:02d}' for i in range(CHOSEN_COMPONENTS)]
item_factors_pl = pl.DataFrame(
    {'movieId': unique_movies} | {col: V_factors[:, i].tolist() for i, col in enumerate(factor_cols)}
)
item_factors_pl.write_parquet(ARTIFACTS_DIR / 'week10_svd_item_factors.parquet')
print(f'Saved item factors: {item_factors_pl.shape}')

Final SVD: n_components=50, explained_variance=0.1779, time=2.9s
U_factors shape (users): (162540, 50)
V_factors shape (items): (13176, 50)
Saved item factors: (13176, 51)


## 4) Generate SVD-based recommendations

For each query movie, compute cosine similarity between its item factor and all other item factors.
This is equivalent to collaborative-filtering item-to-item recommendation:
movies with similar latent factor profiles attract similar users.

In [7]:
TOP_N = 20

# Query set: same 5,000 movies as baseline notebook (top-rated)
rating_agg = (
    ratings_pl.read_parquet(DATA_DIR / 'ratings_clean.parquet')
    if False else  # already loaded above
    ratings_pl
)
movie_pop = (
    rating_agg
    .group_by('movieId')
    .agg(pl.len().alias('rating_count'))
    .filter(pl.col('movieId').is_in(unique_movies))
    .sort('rating_count', descending=True)
)
query_movie_ids = movie_pop.head(5000)['movieId'].to_list()
print(f'Query movies for SVD evaluation: {len(query_movie_ids):,}')

# Build index lookup
svd_recs_rows = []
for qid in query_movie_ids:
    if qid not in movie_to_idx:
        continue
    q_idx = movie_to_idx[qid]
    q_vec = V_normed[q_idx].reshape(1, -1)
    scores = (V_normed @ q_vec.T).flatten()
    sorted_idxs = np.argsort(-scores)
    rank = 0
    for idx in sorted_idxs:
        mid = idx_to_movie[idx]
        if mid == qid:
            continue
        rank += 1
        svd_recs_rows.append({
            'query_movieId': qid,
            'rec_movieId': mid,
            'rank': rank,
            'svd_score': float(scores[idx]),
        })
        if rank >= TOP_N:
            break

svd_recs_pl = pl.DataFrame(svd_recs_rows)
svd_recs_pl.write_parquet(ARTIFACTS_DIR / 'week10_svd_recs_top20.parquet')
print(f'Saved {svd_recs_pl.height:,} SVD recommendation rows.')
svd_recs_pl.head(10)

Query movies for SVD evaluation: 5,000
Saved 100,000 SVD recommendation rows.


query_movieId,rec_movieId,rank,svd_score
i64,i64,i64,f64
356,84844,1,0.355602
356,2685,2,0.3237
356,25906,3,0.299049
356,102602,4,0.294183
356,5251,5,0.277969
356,54094,6,0.239552
356,145994,7,0.228919
356,6390,8,0.226155
356,1725,9,0.22411


## 5) SVD score distribution

Distribution of cosine similarity scores (SVD item factors, rank=1) across all query movies.
Compared to content-based similarity, SVD scores capture behavioral co-occurrence patterns.

In [8]:
top1_scores = svd_recs_pl.filter(pl.col('rank') == 1)['svd_score'].to_list()

fig_svd_dist = go.Figure()
fig_svd_dist.add_trace(go.Histogram(
    x=top1_scores,
    nbinsx=60,
    marker_color='#f59e0b',
    marker_line_color='#d97706',
    marker_line_width=0.5,
    opacity=0.85,
    name='Top-1 SVD score',
))
fig_svd_dist.update_layout(
    title='Distribution of top-1 SVD similarity score (collaborative filtering)',
    xaxis_title='Cosine similarity (SVD item factors)',
    yaxis_title='Number of query movies',
    height=450,
    template='plotly_white',
    bargap=0.05,
)
fig_svd_dist.write_html(ARTIFACTS_DIR / 'week10_svd_score_dist.html')
fig_svd_dist.write_image(ARTIFACTS_DIR / 'week10_svd_score_dist.png', scale=2)
fig_svd_dist.show()

## 6) Singular value spectrum

The singular value spectrum shows how much variance each latent factor captures.
A fast decay indicates the matrix has strong low-rank structure (good for collaborative filtering).
A slow decay means the rating matrix is noisy and harder to approximate.

In [14]:
singular_values = svd_final.singular_values_
singular_variance_ratio = svd_final.explained_variance_ratio_

sv_df = pl.DataFrame({
    'component': list(range(1, CHOSEN_COMPONENTS + 1)),
    'singular_value': singular_values.tolist(),
    'explained_variance_ratio': singular_variance_ratio.tolist(),
    'cumulative_variance': np.cumsum(singular_variance_ratio).tolist(),
})
sv_df.write_csv(ARTIFACTS_DIR / 'week10_svd_singular_values.csv')

fig_sv = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Singular value spectrum', 'Cumulative explained variance'),
)
fig_sv.add_trace(
    go.Bar(
        x=sv_df['component'].to_list(),
        y=sv_df['singular_value'].to_list(),
        marker_color='#8b5cf6',
        name='Singular value',
        showlegend=False,
    ),
    row=1, col=1,
)
fig_sv.add_trace(
    go.Scatter(
        x=sv_df['component'].to_list(),
        y=sv_df['cumulative_variance'].to_list(),
        mode='lines+markers',
        marker=dict(size=4, color='#ec4899'),
        line=dict(width=2, color='#ec4899'),
        name='Cumulative variance',
        showlegend=False,
    ),
    row=1, col=2,
)
fig_sv.update_xaxes(title_text='Component', row=1, col=1)
fig_sv.update_xaxes(title_text='Component', row=1, col=2)
fig_sv.update_yaxes(title_text='Singular value', row=1, col=1)
fig_sv.update_yaxes(title_text='Cumulative variance ratio', row=1, col=2)
fig_sv.update_layout(
    title='SVD singular value analysis',
    height=450,
    width=1200,
    template='plotly_white',
)
fig_sv.write_html(ARTIFACTS_DIR / 'week10_svd_singular_values.html')
fig_sv.write_image(ARTIFACTS_DIR / 'week10_svd_singular_values.png', scale=2)
fig_sv.show()

## 7) Qualitative check — recommendation examples

Manual inspection of SVD recommendations for known movies.
This is not evaluation but it helps catch obvious failures:
if Toy Story recommends action thrillers, something is wrong with the factorization.

In [10]:
demo_ids = [
    1,      # Toy Story (1995)
    296,    # Pulp Fiction (1994)
    318,    # Shawshank Redemption (1994)
    2571,   # Matrix (1999)
]

svd_demo_rows = []
for qid in demo_ids:
    recs_subset = (
        svd_recs_pl
        .filter(pl.col('query_movieId') == qid)
        .sort('rank')
        .head(5)
    )
    for row in recs_subset.iter_rows(named=True):
        svd_demo_rows.append({
            'query_id': qid,
            'query_title': title_map.get(qid, '?'),
            'rank': row['rank'],
            'rec_movieId': row['rec_movieId'],
            'rec_title': title_map.get(row['rec_movieId'], '?'),
            'svd_score': round(row['svd_score'], 4),
        })

svd_demo_df = pd.DataFrame(svd_demo_rows)
print('SVD collaborative filtering recommendations (demo):')
display(svd_demo_df)

SVD collaborative filtering recommendations (demo):


,query_id,query_title,rank,rec_movieId,rec_title,svd_score
0,1,Toy Story (1995),1,3114,Toy Story 2 (1999),0.8553
1,1,Toy Story (1995),2,78499,Toy Story 3 (2010),0.5947
2,1,Toy Story (1995),3,2355,"Bug's Life, A (1998)",0.5180
3,1,Toy Story (1995),4,120474,Toy Story That Time Forgot (2014),0.4197
4,1,Toy Story (1995),5,192697,Destination Wedding (2018),0.4072
5,296,Pulp Fiction (1994),1,1089,Reservoir Dogs (1992),0.2948
6,296,Pulp Fiction (1994),2,21,Get Shorty (1995),0.2817
7,296,Pulp Fiction (1994),3,115965,Dead Man's Bluff (2005),0.2470
8,296,Pulp Fiction (1994),4,146604,Naomi and Ely's No Kiss List (2015),0.2469
9,296,Pulp Fiction (1994),5,3575,Defying Gravity (1997),0.2354


## 8) Save model metadata

In [11]:
svd_meta = {
    'model': 'TruncatedSVD_item_factors',
    'library': 'sklearn.decomposition.TruncatedSVD',
    'min_movie_ratings': MIN_MOVIE_RATINGS,
    'min_user_ratings': MIN_USER_RATINGS,
    'n_users_in_matrix': n_users,
    'n_movies_in_matrix': n_movies,
    'matrix_density': round(density, 8),
    'n_components_chosen': CHOSEN_COMPONENTS,
    'explained_variance_chosen': round(explained_final, 6),
    'rating_centering': 'per_user_mean',
    'similarity_metric': 'cosine (L2-normalized item factors)',
    'top_n': TOP_N,
    'n_query_movies': len(query_movie_ids),
    'artifacts': [
        'week10_svd_item_factors.parquet',
        'week10_svd_recs_top20.parquet',
        'week10_svd_sweep.csv',
        'week10_svd_singular_values.csv',
    ],
}

with open(ARTIFACTS_DIR / 'week10_svd_meta.json', 'w') as f:
    json.dump(svd_meta, f, indent=2)

print('SVD model metadata saved.')
print(json.dumps(svd_meta, indent=2))

SVD model metadata saved.
{
  "model": "TruncatedSVD_item_factors",
  "library": "sklearn.decomposition.TruncatedSVD",
  "min_movie_ratings": 50,
  "min_user_ratings": 20,
  "n_users_in_matrix": 162540,
  "n_movies_in_matrix": 13176,
  "matrix_density": 0.01150757,
  "n_components_chosen": 50,
  "explained_variance_chosen": 0.177907,
  "rating_centering": "per_user_mean",
  "similarity_metric": "cosine (L2-normalized item factors)",
  "top_n": 20,
  "n_query_movies": 5000,
  "artifacts": [
    "week10_svd_item_factors.parquet",
    "week10_svd_recs_top20.parquet",
    "week10_svd_sweep.csv",
    "week10_svd_singular_values.csv"
  ]
}
